# Decision Trees: Readable Rules Without Overfitting
### Course template: Machine Learning for Business Insights (Module 5)

**What this notebook does**
1. Loads your CSV and checks data quality
2. Builds a leakage-safe pipeline
3. Trains a baseline and a default (unlimited) decision tree, and shows how badly it overfits
4. Tunes `max_depth`, `min_samples_leaf`, and related controls with cross-validation
5. Evaluates with business metrics (classification or regression)
6. Prints plain-English **IF–THEN rules** and feature importance

**Works for both:**
- **Classification** (yes/no: late payment, churn, conversion)
- **Regression** (a number: job cost, delivery days, order value)

Edit **only the CONFIG cell**, then run top to bottom. Leave `CSV_PATH = None` to try the demo first.

In [ ]:
# ============================================================
# IMPORTS (all preinstalled in Colab; needs scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, RepeatedKFold, RepeatedStratifiedKFold,
    cross_validate, cross_val_predict, GridSearchCV, RandomizedSearchCV, learning_curve, validation_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
RANDOM_STATE = 42
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: EDIT THIS CELL
# ============================================================

# --- 1. Task type --------------------------------------------
TASK = "classification"     # "classification" (yes/no) or "regression" (a number)

# --- 2. Data -------------------------------------------------
CSV_PATH = None             # e.g. "/content/invoices.csv"; None = demo data

# --- 3. Target -----------------------------------------------
TARGET_COLUMN = "paid_late" if TASK == "classification" else "job_cost"
POSITIVE_VALUE = 1          # classification only: the value that means "YES"

# --- 4. Inputs -----------------------------------------------
NUMERIC_COLUMNS = (["invoice_amount", "customer_tenure_months", "past_late_payments", "days_to_due"]
                   if TASK == "classification" else
                   ["sq_feet", "crew_size", "distance_miles", "rush_days"])
CATEGORICAL_COLUMNS = (["industry", "payment_method"]
                       if TASK == "classification" else
                       ["material", "job_type"])

# IDs and LEAKAGE columns (anything not known on decision day).
# Leakage example: "days_overdue" when predicting whether an invoice will be late.
DROP_COLUMNS = ["record_id"]

# --- 5. Classification settings -----------------------------
MAIN_METRIC_CLF = "roc_auc"   # "roc_auc", "average_precision" (rare YES), "f1", "recall"
VALUE_IF_CAUGHT = 400         # value of acting correctly on one YES case
ACTION_COST = 60              # cost of acting on one flagged case

# --- 6. General ----------------------------------------------
DATA_SIZE_MODE = "auto"       # "auto", "small", "large"
TUNING_METHOD = "grid"        # "grid" (small) or "random" (large)
SAMPLE_ROWS_FOR_EXPLORATION = None
TEST_SIZE = 0.2
UNIT_LABEL = "$"              # regression: "$", "days", "units", ...

IS_CLF = TASK == "classification"
print(f"Task: {TASK} | Target: {TARGET_COLUMN}")

In [ ]:
# ============================================================
# LOAD DATA. Demo data is SYNTHETIC; don't draw real conclusions.
# ============================================================

def make_demo_clf(n=400, seed=RANDOM_STATE):
    """Invoices: will this be paid late? Built with clear thresholds a tree can find."""
    rng = np.random.default_rng(seed)
    amount = rng.lognormal(8, 0.8, n).round(0)
    tenure = rng.integers(1, 120, n)
    past_late = rng.poisson(0.8, n)
    days_due = rng.choice([15, 30, 45, 60], n)
    industry = rng.choice(["retail", "construction", "healthcare", "tech"], n)
    method = rng.choice(["ach", "check", "card"], n, p=[0.5, 0.3, 0.2])
    p = (0.08 + 0.30 * (past_late >= 2) + 0.20 * ((amount > 5_000) & (tenure < 12))
         + 0.10 * (method == "check") + 0.08 * (industry == "construction"))
    late = (rng.random(n) < p.clip(0, 0.95)).astype(int)
    df = pd.DataFrame({"record_id": range(n), "invoice_amount": amount, "customer_tenure_months": tenure,
                       "past_late_payments": past_late, "days_to_due": days_due,
                       "industry": industry, "payment_method": method, "paid_late": late})
    df.loc[rng.choice(n, 10, replace=False), "customer_tenure_months"] = np.nan
    return df

def make_demo_reg(n=150, seed=RANDOM_STATE):
    """Service jobs: what will this job cost?"""
    rng = np.random.default_rng(seed)
    sq = rng.integers(200, 3_000, n)
    crew = rng.integers(1, 6, n)
    dist = rng.gamma(2, 8, n).round(1)
    rush = rng.choice([0, 1, 3, 7], n, p=[0.5, 0.2, 0.2, 0.1])
    material = rng.choice(["standard", "premium", "budget"], n, p=[0.5, 0.25, 0.25])
    jtype = rng.choice(["repair", "install", "replace"], n)
    cost = (800 + 3.5 * sq + 300 * crew + 15 * dist
            + np.where(rush > 0, 1_200, 0)
            + np.select([material == "premium", material == "budget"], [2_500, -600], 0)
            + np.where(jtype == "replace", 1_500, 0) + rng.normal(0, 900, n))
    return pd.DataFrame({"record_id": range(n), "sq_feet": sq, "crew_size": crew, "distance_miles": dist,
                         "rush_days": rush, "material": material, "job_type": jtype,
                         "job_cost": cost.round(0)})

if CSV_PATH is None:
    df = make_demo_clf() if IS_CLF else make_demo_reg()
    print("Using DEMO data (synthetic).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")

if SAMPLE_ROWS_FOR_EXPLORATION and len(df) > SAMPLE_ROWS_FOR_EXPLORATION:
    df = df.sample(SAMPLE_ROWS_FOR_EXPLORATION, random_state=RANDOM_STATE)
    print("Sampled rows for exploration. Set back to None for final results.")

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK + TARGET
# ============================================================
cols = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"Not found in data: {missing}\nYour columns: {list(df.columns)}")

for c in NUMERIC_COLUMNS + ([] if IS_CLF else [TARGET_COLUMN]):
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[$,%\s]", "", regex=True), errors="coerce")
        print(f"Converted '{c}' to numbers.")

df = df.dropna(subset=[TARGET_COLUMN])

if IS_CLF:
    df["_y"] = (df[TARGET_COLUMN].astype(str).str.strip().str.lower()
                == str(POSITIVE_VALUE).strip().lower()).astype(int)
    if df["_y"].nunique() < 2:
        raise ValueError("Target has one class after mapping. Check POSITIVE_VALUE.")
else:
    df["_y"] = df[TARGET_COLUMN].astype(float)
print("Target ready.")

## Section 1: Quick EDA and Data Quality
Look for blanks, impossible values, duplicates, category typos, and (for classification) how rare YES is.
Trees are sensitive to **rare categories** and **ID-like columns**: they can split on them and "memorize" rows.

In [ ]:
# ============================================================
# DATA QUALITY REPORT
# ============================================================
print("=== Missing values ===")
m = df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS].isna().sum()
print(m[m > 0] if m.sum() else "None", "\n")
print("Duplicate rows:", df.duplicated().sum(), "\n")
display(df[NUMERIC_COLUMNS].describe().T)

for c in CATEGORICAL_COLUMNS:
    vc = df[c].value_counts(dropna=False)
    print(f"\n{c} ({vc.size} values):"); print(vc)
    if (vc < 10).any():
        print(f"  ⚠ Rare values in {c} (<10 rows). Consider grouping into 'Other'.")

# ID-like numeric columns (almost every value unique) are a tree overfitting trap
for c in NUMERIC_COLUMNS:
    if df[c].nunique() > 0.95 * len(df) and df[c].dtype.kind in "i":
        print(f"⚠ {c} looks like an ID (nearly all values unique). Should it be in DROP_COLUMNS?")

if IS_CLF:
    yes = df["_y"].mean()
    print(f"\nYES rate: {yes:.1%} ({int(df['_y'].sum())} of {len(df)})")
else:
    df["_y"].hist(bins=30, figsize=(6, 3)); plt.title(f"Distribution of {TARGET_COLUMN}"); plt.show()

In [ ]:
# ============================================================
# DATA SIZE CHECK
# Trees need MORE data than linear models to be stable, because every
# split cuts the data into smaller groups. With 200 rows, a depth-4 tree
# can have 16 leaves averaging just ~12 rows each.
# ============================================================
n = len(df)
n_minority = int(min(df["_y"].sum(), n - df["_y"].sum())) if IS_CLF else n
key_count = n_minority if IS_CLF else n
MODE = ("small" if key_count < (200 if IS_CLF else 500) else "large") if DATA_SIZE_MODE == "auto" else DATA_SIZE_MODE

print(f"Rows: {n:,}" + (f" | Rare-group cases: {n_minority:,}" if IS_CLF else ""))
print(f"Mode: {MODE.upper()}")
for depth in [2, 3, 4, 6]:
    print(f"  depth {depth}: up to {2**depth} leaves → ~{n * (1 - TEST_SIZE) / 2**depth:,.0f} training rows per leaf")
if MODE == "small":
    print("\nSMALL-DATA PATH: depth capped at 2–5, large minimum leaf sizes, repeated CV.")

## Section 2: Split and Validation
- The **test set** is locked away until the end.
- Classification uses **stratified** splits so every fold has a fair share of YES.
- Small data uses **repeated** CV, because a tree's score on one split can swing a lot.

In [ ]:
# ============================================================
# SPLIT + CV
# ============================================================
feature_cols = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X, y = df[feature_cols], df["_y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y if IS_CLF else None)

if IS_CLF:
    CV = (RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE) if MODE == "small"
          else StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
else:
    CV = (RepeatedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE) if MODE == "small"
          else KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
print(f"Train {len(X_train):,} | Test {len(X_test):,} | CV: {type(CV).__name__}")

## Section 3: Leakage-Safe Pipeline (Tree Version)

Trees are simpler to prep than linear models:
- **No scaling needed.** A tree only asks "is X above or below some value?"
- **Ordinal encoding is fine** for categories. The tree can still separate any group. (One-hot works too but makes rules longer.)
- Blanks are filled **inside** the pipeline, so fill values are learned only from training folds.

In [ ]:
# ============================================================
# PREPROCESSOR
# ============================================================
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_COLUMNS),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        # Unknown categories in future data become -1 instead of crashing
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CATEGORICAL_COLUMNS),
], verbose_feature_names_out=False)   # keeps clean column names for rules

Tree = DecisionTreeClassifier if IS_CLF else DecisionTreeRegressor
Dummy = (lambda: DummyClassifier(strategy="most_frequent")) if IS_CLF else (lambda: DummyRegressor())

def make_model(est):
    return Pipeline([("prep", preprocessor), ("model", est)])
print("Pipeline ready.")

## Section 4: Baseline vs. Default Tree (Watch It Overfit)

A **default** decision tree has no limits: it keeps splitting until every leaf is pure.
That usually means **near-perfect training scores and much worse validation scores**. That gap *is* overfitting.

We also train a **depth-3 tree** as a simple, readable reference.

In [ ]:
# ============================================================
# BASELINE vs. DEFAULT vs. SHALLOW TREE
# return_train_score=True lets us SEE overfitting: train vs. validation.
# ============================================================
if IS_CLF:
    SCORING = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision", "F1": "f1",
               "Precision": "precision", "Recall": "recall"}
    MAIN_METRIC = MAIN_METRIC_CLF
else:
    SCORING = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
    MAIN_METRIC = "neg_mean_absolute_error"

candidates = {
    "Baseline": make_model(Dummy()),
    "Default tree (no limits)": make_model(Tree(random_state=RANDOM_STATE)),
    "Shallow tree (depth 3)": make_model(Tree(max_depth=3, min_samples_leaf=10, random_state=RANDOM_STATE)),
}

def cv_summary(model, name):
    r = cross_validate(model, X_train, y_train, cv=CV, scoring=SCORING, return_train_score=True, error_score=np.nan)
    out = {"model": name}
    for k in SCORING:
        sign = -1 if SCORING[k].startswith("neg_") else 1
        out[f"{k} (valid)"] = sign * np.nanmean(r[f"test_{k}"])
        out[f"{k} ±"] = np.nanstd(r[f"test_{k}"])
        out[f"{k} (train)"] = sign * np.nanmean(r[f"train_{k}"])
    return out

res = pd.DataFrame([cv_summary(m, n) for n, m in candidates.items()]).set_index("model")
display(res)

key = "ROC-AUC" if IS_CLF else "MAE"
print(f"""
HOW TO READ THIS
- Compare "(train)" with "(valid)". A big gap = the model memorized the training data.
  The default tree usually scores near-perfect on train ({key} train) but much worse on validation.
- ± = spread across folds. Differences smaller than the spread are effectively ties.
- If the shallow tree's validation score is close to the default tree's, the shallow tree
  wins: same accuracy, far more readable and stable.
""")

## Section 5: Hyperparameter Tuning (Overfitting Control)

These settings stop the tree from memorizing noise. In plain terms:

| Setting | What it controls | Small data | Large data |
|---|---|---|---|
| `max_depth` | How many questions deep the flowchart goes | 2–5 | 3–12 |
| `min_samples_leaf` | Smallest group allowed at the end of a branch | Large (5–10% of rows) | 5–100 |
| `min_samples_split` | Smallest group allowed to be split further | 2× leaf size | 10–200 |
| `max_leaf_nodes` | Hard cap on number of final groups | 4–12 | 8–64 |
| `ccp_alpha` | "Pruning": removes branches that add little | Try a few small values | Try a few small values |
| `class_weight` (clf) | Gives rare YES cases more weight | None / balanced | None / balanced |

⚠️ **Small data: the most important warning in this notebook.** Every split halves your data.
A leaf with 6 rows reporting "83% late payments" is 5 invoices out of 6. That's not a pattern you
should run a business on. Keep trees **shallow** and leaves **big**, and prefer the simplest tree
within one spread of the best score.

In [ ]:
# ============================================================
# VALIDATION CURVE: score vs. depth
# Shows the classic overfitting pattern: training keeps improving with
# depth while validation peaks and then flattens or drops.
# ============================================================
depths = list(range(1, 13 if MODE == "large" else 9))
tr, va = validation_curve(
    make_model(Tree(random_state=RANDOM_STATE)), X_train, y_train,
    param_name="model__max_depth", param_range=depths, cv=CV, scoring=MAIN_METRIC)
sign = -1 if MAIN_METRIC.startswith("neg_") else 1
plt.figure(figsize=(7, 4))
plt.plot(depths, sign * tr.mean(1), "o-", label="Training")
plt.errorbar(depths, sign * va.mean(1), yerr=va.std(1), fmt="o-", capsize=3, label="Validation")
plt.xlabel("max_depth"); plt.ylabel(MAIN_METRIC.replace("neg_", ""))
plt.title("Deeper trees: better on training, not on new data"); plt.legend(); plt.show()

In [ ]:
# ============================================================
# TUNING SEARCH
# Leaf sizes are set as FRACTIONS of training rows on small data so
# the grid adapts to your dataset size automatically.
# ============================================================
n_tr = len(X_train)
if MODE == "small":
    leaf_opts = sorted({max(5, int(n_tr * f)) for f in [0.03, 0.05, 0.08, 0.12]})
    grid = {
        "model__max_depth": [2, 3, 4, 5],
        "model__min_samples_leaf": leaf_opts,
        "model__ccp_alpha": [0.0, 0.001, 0.005] if IS_CLF else [0.0],
    }
else:
    grid = {
        "model__max_depth": [3, 4, 5, 6, 8, 10, 12],
        "model__min_samples_leaf": [5, 10, 20, 50, 100],
        "model__min_samples_split": [10, 20, 50, 100],
        "model__max_leaf_nodes": [None, 16, 32, 64],
        "model__ccp_alpha": [0.0, 0.0005, 0.001, 0.005] if IS_CLF else [0.0],
    }
if IS_CLF:
    grid["model__class_weight"] = [None, "balanced"]

n_combos = int(np.prod([len(v) for v in grid.values()]))
method = TUNING_METHOD
if method == "grid" and n_combos > 600:
    method = "random"   # a full grid this size would be slow; sample it instead
    print(f"Grid has {n_combos:,} combinations → switching to randomized search for speed.")

base = make_model(Tree(random_state=RANDOM_STATE))
if method == "random":
    search = RandomizedSearchCV(base, grid, n_iter=30 if MODE == "small" else 60, cv=CV,
                                scoring=MAIN_METRIC, random_state=RANDOM_STATE, n_jobs=-1)
else:
    search = GridSearchCV(base, grid, cv=CV, scoring=MAIN_METRIC, n_jobs=-1)
search.fit(X_train, y_train)

# Results table (one row per setting tried). "_i" remembers the original row.
cvr = search.cv_results_
tbl = pd.DataFrame(cvr["params"]).rename(columns=lambda c: c.replace("model__", ""))
tbl["score"] = sign * cvr["mean_test_score"]        # errors shown as positive numbers
tbl["±"] = cvr["std_test_score"]
tbl["_i"] = range(len(tbl))
HIGHER_IS_BETTER = sign == 1
tbl = tbl.sort_values("score", ascending=not HIGHER_IS_BETTER, kind="stable").reset_index(drop=True)
print(f"Top 10 settings ({MAIN_METRIC.replace('neg_', '')}):")
top = tbl.drop(columns="_i").head(10).copy()
for c in top.columns[:-2]:                            # show "None" instead of NaN for unset options
    top[c] = top[c].astype(object).where(top[c].notna(), "None").astype(str)
display(top)

# ============================================================
# SIMPLICITY RULE: among settings within one spread of the best,
# pick the SHALLOWEST tree, then the BIGGEST leaves.
# Shallow + big leaves = fewer, sturdier rules you can explain.
# ============================================================
best = tbl.iloc[0]
if HIGHER_IS_BETTER:
    within = tbl[tbl["score"] >= best["score"] - best["±"]].copy()
else:
    within = tbl[tbl["score"] <= best["score"] + best["±"]].copy()
within["_depth"] = within["max_depth"].fillna(99)
chosen = within.sort_values(["_depth", "min_samples_leaf"], ascending=[True, False]).iloc[0]
BEST_PARAMS = {k.replace("model__", ""): v for k, v in cvr["params"][int(chosen["_i"])].items()}

print(f"\nBest score: {best['score']:.3f} ± {best['±']:.3f}")
print(f"{len(within)} settings are within one spread of the best.")
print(f"CHOSEN (shallowest, then biggest leaves): {BEST_PARAMS} → score {chosen['score']:.3f}")

In [ ]:
# ============================================================
# CHOSEN TREE vs. BASELINE / DEFAULT / SHALLOW (same CV folds)
# ============================================================
final_model = make_model(Tree(random_state=RANDOM_STATE, **BEST_PARAMS))
res2 = pd.concat([res, pd.DataFrame([cv_summary(final_model, "Tuned tree (chosen)")]).set_index("model")])
KEY = {v: k for k, v in SCORING.items()}[MAIN_METRIC]      # e.g. "ROC-AUC" or "MAE"
display(res2[[f"{KEY} (valid)", f"{KEY} ±", f"{KEY} (train)"]])

gap_default = abs(res2.loc["Default tree (no limits)", f"{KEY} (train)"] - res2.loc["Default tree (no limits)", f"{KEY} (valid)"])
gap_tuned = abs(res2.loc["Tuned tree (chosen)", f"{KEY} (train)"] - res2.loc["Tuned tree (chosen)", f"{KEY} (valid)"])
print(f"Train-vs-validation gap: default tree {gap_default:.3f} → tuned tree {gap_tuned:.3f}")
print("A smaller gap means the tuned tree learned patterns instead of memorizing rows.")

## Section 6: Final Test-Set Evaluation

We fit the chosen tree on all training rows and score the locked test set **once**.

**Classification:** a tree gives each leaf a YES rate, so its "probabilities" come in just a few distinct
values. We pick the action threshold from **out-of-fold** predictions on the training data using your
`VALUE_IF_CAUGHT` and `ACTION_COST`, then check it on the test set.

**Regression:** we report the typical miss in business units against the "predict the average" baseline.

In [ ]:
# ============================================================
# TEST-SET EVALUATION
# ============================================================
final_model.fit(X_train, y_train)

if IS_CLF:
    proba_test = final_model.predict_proba(X_test)[:, 1]
    print(f"TEST ROC-AUC: {roc_auc_score(y_test, proba_test):.3f}   (0.5 = coin flip)")
    print(f"TEST PR-AUC:  {average_precision_score(y_test, proba_test):.3f}   (random = {y_test.mean():.3f})")

    # --- Threshold from out-of-fold predictions (never the test set) ---
    OOF_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    proba_oof = cross_val_predict(final_model, X_train, y_train, cv=OOF_CV, method="predict_proba")[:, 1]
    rows = []
    for t in np.unique(np.round(proba_oof, 3)):
        flag = proba_oof >= t
        tp = int((flag & (y_train.values == 1)).sum()); nf = int(flag.sum())
        rows.append({"threshold": t, "flagged": nf, "YES caught": tp,
                     "precision": tp / nf if nf else np.nan, "recall": tp / y_train.sum(),
                     "expected value": tp * VALUE_IF_CAUGHT - nf * ACTION_COST})
    tt = pd.DataFrame(rows)
    THRESHOLD = float(tt.loc[tt["expected value"].idxmax(), "threshold"])
    display(tt.style.format({"precision": "{:.2f}", "recall": "{:.2f}", "expected value": "${:,.0f}"}).hide(axis="index"))
    print(f"Break-even precision: {ACTION_COST / VALUE_IF_CAUGHT:.1%} (ACTION_COST ÷ VALUE_IF_CAUGHT)")
    print(f"CHOSEN THRESHOLD (best expected value on out-of-fold predictions): {THRESHOLD:.3f}")

    pred = (proba_test >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, pred)
    print("\nTEST SET at the chosen threshold:")
    print(pd.DataFrame(cm, index=["Actual NO", "Actual YES"], columns=["Flagged NO", "Flagged YES"]))
    tp, nf = cm[1, 1], cm[:, 1].sum()
    print(f"Precision {precision_score(y_test, pred, zero_division=0):.2f} | Recall {recall_score(y_test, pred):.2f} | "
          f"F1 {f1_score(y_test, pred):.2f}")
    print(f"Estimated value on the test set: ${tp * VALUE_IF_CAUGHT - nf * ACTION_COST:,.0f} "
          f"(acting on {nf} of {len(y_test)} cases)")
else:
    pred = final_model.predict(X_test)
    base_pred = np.full(len(y_test), y_train.mean())
    mae, mae_b = mean_absolute_error(y_test, pred), mean_absolute_error(y_test, base_pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    display(pd.DataFrame({"Baseline (average)": [mae_b, np.sqrt(mean_squared_error(y_test, base_pred)), r2_score(y_test, base_pred)],
                          "Tuned tree": [mae, rmse, r2_score(y_test, pred)]}, index=["MAE", "RMSE", "R2"]))
    print(f"""
IN PLAIN TERMS
- The tree's typical miss is {UNIT_LABEL}{mae:,.0f} ({mae / y_test.mean():.0%} of the average {TARGET_COLUMN}).
- Guessing the average misses by {UNIT_LABEL}{mae_b:,.0f}, so the tree cuts the typical miss by {1 - mae / mae_b:.0%}.
- A tree predicts one fixed value per leaf, so forecasts come in "steps". If you need smoother
  estimates, compare with Linear Regression (Module 3) or Random Forest / Gradient Boosting.""")
    plt.figure(figsize=(5, 5)); plt.scatter(y_test, pred, alpha=0.6)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    plt.plot(lims, lims, "r--"); plt.xlabel("Actual"); plt.ylabel("Predicted (leaf value)")
    plt.title("Actual vs. predicted (test)"); plt.show()

## Section 7: The Rules (Why Decision Trees Are Worth It)

The tree is a flowchart. Below you'll see:
1. A **picture** of the tree
2. The **text version** (`export_text`)
3. A **rules table**: one row per final group (leaf), in plain English, with how many training rows
   back it up. Leaves marked ⚠ are small: treat them as hints, not policy.

Category splits are translated back into names, e.g. "payment_method is check / card".

In [ ]:
# ============================================================
# TREE PICTURE + TEXT RULES
# ============================================================
tree = final_model.named_steps["model"]
prep = final_model.named_steps["prep"]
feat = list(prep.get_feature_names_out())
cat_map = dict(zip(CATEGORICAL_COLUMNS, prep.named_transformers_["cat"].named_steps["ord"].categories_))

plt.figure(figsize=(min(4 + 3 * tree.get_n_leaves(), 26), 3 + 1.8 * tree.get_depth()))
plot_tree(tree, feature_names=feat, class_names=["NO", "YES"] if IS_CLF else None,
          filled=True, rounded=True, impurity=False, proportion=True, fontsize=9)
plt.title("Decision tree (category values are coded; see the code list below)"); plt.show()

for c, cats in cat_map.items():
    print(f"Codes for {c}: " + ", ".join(f"{i}={v}" for i, v in enumerate(cats)))
print()
print(export_text(tree, feature_names=feat, decimals=2))
if IS_CLF:
    print("Note: 'class' above is the MAJORITY outcome in each leaf. A leaf can say 'class: 0' and still have a\n"
          "high YES rate. The rules table below shows the actual YES rate per leaf, which is what matters.")

In [ ]:
# ============================================================
# PLAIN-ENGLISH LEAF RULES TABLE
# Walks every path from the top of the tree to each leaf, combining
# the conditions (and translating category codes back into names).
# ============================================================
t = tree.tree_
paths = {}

def walk(node, bounds, allowed):
    if t.children_left[node] == -1:                      # reached a leaf
        paths[node] = (bounds, allowed); return
    name, thr = feat[t.feature[node]], t.threshold[node]
    for go_left, child in [(True, t.children_left[node]), (False, t.children_right[node])]:
        b, a = dict(bounds), {k: set(v) for k, v in allowed.items()}
        if name in cat_map:
            cats = cat_map[name]
            keep = {c for i, c in enumerate(cats) if (i <= thr) == go_left}
            a[name] = a.get(name, set(cats)) & keep
        else:
            lo, hi = b.get(name, (-np.inf, np.inf))
            b[name] = (lo, min(hi, thr)) if go_left else (max(lo, thr), hi)
        walk(child, b, a)

walk(0, {}, {})

def rule_text(bounds, allowed):
    parts = []
    for name, (lo, hi) in bounds.items():
        if lo == -np.inf:   parts.append(f"{name} ≤ {hi:,.2f}")
        elif hi == np.inf:  parts.append(f"{name} > {lo:,.2f}")
        else:               parts.append(f"{name} between {lo:,.2f} and {hi:,.2f}")
    for name, vals in allowed.items():
        parts.append(f"{name} is {' / '.join(sorted(map(str, vals)))}")
    return " AND ".join(parts) if parts else "All rows"

# Leaf statistics from the actual training rows in each leaf
leaf_of_row = tree.apply(final_model[:-1].transform(X_train))
stats = pd.DataFrame({"leaf": leaf_of_row, "y": y_train.values}).groupby("leaf")["y"].agg(["size", "mean", "sum"])
SMALL_LEAF = max(20, int(0.05 * len(X_train)))
overall = y_train.mean()

rules = pd.DataFrame([{
    "rule (IF ...)": rule_text(*paths[leaf]),
    "training rows": int(stats.loc[leaf, "size"]),
    ("YES rate" if IS_CLF else f"average {TARGET_COLUMN}"): stats.loc[leaf, "mean"],
    "vs. overall": stats.loc[leaf, "mean"] / overall if overall else np.nan,
    "caution": "⚠ small group" if stats.loc[leaf, "size"] < SMALL_LEAF else "",
} for leaf in stats.index])
val_col = rules.columns[2]
rules = rules.sort_values(val_col, ascending=False).reset_index(drop=True)
pd.set_option("display.max_colwidth", 200)
display(rules.style.format({val_col: "{:.1%}" if IS_CLF else "{:,.0f}", "vs. overall": "{:.2f}x"}))

print("PLAIN-ENGLISH RULES:")
for _, r in rules.iterrows():
    v = f"{r[val_col]:.0%} YES rate" if IS_CLF else f"average {UNIT_LABEL}{r[val_col]:,.0f}"
    ref = f"{overall:.0%}" if IS_CLF else f"{UNIT_LABEL}{overall:,.0f}"
    print(f"- IF {r['rule (IF ...)']}\n    THEN {v} (overall {ref}), based on {r['training rows']} rows {r['caution']}")

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# Permutation importance (preferred): shuffle one input at a time on
# the TEST set and measure how much the score drops. Inputs the model
# truly relies on cause big drops.
# Built-in importance: how much each input's splits reduced impurity
# on TRAINING data. Fast, but can overrate inputs with many values.
# ============================================================
perm = permutation_importance(final_model, X_test, y_test, scoring=MAIN_METRIC,
                              n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({
    "permutation (score drop)": perm.importances_mean,
    "±": perm.importances_std,
    "built-in (impurity)": pd.Series(tree.feature_importances_, index=feat).reindex(feature_cols).values,
}, index=feature_cols).sort_values("permutation (score drop)", ascending=False)
display(imp)

fig, ax = plt.subplots(1, 2, figsize=(12, 0.45 * len(imp) + 2))
imp["permutation (score drop)"][::-1].plot.barh(ax=ax[0], xerr=imp["±"][::-1], color="steelblue")
ax[0].set_title("Permutation importance (test set)")
imp["built-in (impurity)"][::-1].plot.barh(ax=ax[1], color="gray"); ax[1].set_title("Built-in importance (training)")
plt.tight_layout(); plt.show()
unused = imp.index[imp["built-in (impurity)"] == 0].tolist()
print("Inputs the tree never uses:", unused or "none", "(that's normal for a shallow tree)")

In [ ]:
# ============================================================
# STABILITY CHECK: does the tree tell the same story on resampled data?
# We refit the chosen tree on many bootstrap samples (rows drawn with
# replacement) and record which input it splits on FIRST.
# If the first question changes often, don't present the rules as fact.
# ============================================================
N_BOOT = 100 if MODE == "small" else 30
rng = np.random.default_rng(RANDOM_STATE)
roots = []
for _ in range(N_BOOT):
    idx = rng.integers(0, len(X_train), len(X_train))
    m = clone(final_model).fit(X_train.iloc[idx], y_train.iloc[idx])
    tr_ = m.named_steps["model"].tree_
    roots.append(feat[tr_.feature[0]] if tr_.node_count > 1 else "(no split)")
root_share = pd.Series(roots).value_counts(normalize=True)
print("First split chosen across bootstrap refits:")
print(((root_share * 100).round(0).astype(int).astype(str) + "%").to_string())
top_share = root_share.iloc[0]
if top_share >= 0.8:
    print(f"\n✅ Stable: '{root_share.index[0]}' is the first question {top_share:.0%} of the time.")
else:
    print(f"\n⚠ Unstable: the first question changes a lot ({top_share:.0%} for the most common).\n"
          "  Present the rules as 'one plausible view' and consider a Random Forest for predictions.")

# ============================================================
# DEPTH-2 CHECK: could an even simpler tree do almost as well?
# A 2-question flowchart is easy to put on one slide.
# ============================================================
leaf2 = BEST_PARAMS.get("min_samples_leaf", 1)
d2 = cv_summary(make_model(Tree(max_depth=2, min_samples_leaf=leaf2, random_state=RANDOM_STATE)), "Depth-2 tree")
ch = res2.loc["Tuned tree (chosen)"]
diff = abs(ch[f"{KEY} (valid)"] - d2[f"{KEY} (valid)"])
print(f"\nDepth-2 tree {KEY}: {d2[f'{KEY} (valid)']:.3f} vs. chosen tree {ch[f'{KEY} (valid)']:.3f} (spread ±{ch[f'{KEY} ±']:.3f})")
print("→ Within the spread: the depth-2 tree is a great communication tool." if diff <= ch[f"{KEY} ±"]
      else "→ The deeper tree is meaningfully better; keep it for decisions.")

## Section 8: Small Data vs. Large Data

**Small data (under ~200 YES cases or ~500 rows for numbers):**
- Depth 2–4 and big leaves (≥ 5% of rows) are the norm, and that's healthy
- Every rule should rest on dozens of rows; ignore ⚠ small groups when setting policy
- Run the stability check: if the first split changes, describe rules as hypotheses
- A tuned logistic/linear model often scores as well; the tree's value is the **readable rulebook**

**Large data (thousands of rows and up):**
- Deeper trees (6–12) can be justified, but rules get long; cap `max_leaf_nodes` for readability
- `TUNING_METHOD = "random"` keeps the search fast
- For pure accuracy, Random Forest or Gradient Boosting will usually win; keep the single tree as the explainer
- Rare segments can now support their own rules, so check leaf sizes, not just depth

In [ ]:
# ============================================================
# SAVE THE MODEL AND THE RULES
# ============================================================
import joblib

joblib.dump({"model": final_model, "threshold": THRESHOLD if IS_CLF else None}, "decision_tree_model.joblib")

with open("decision_tree_rules.txt", "w") as f:
    f.write(f"Decision tree rules: target = {TARGET_COLUMN} ({TASK})\n")
    f.write(f"Settings: {BEST_PARAMS}\n\n")
    f.write("Category codes:\n")
    for c, cats in cat_map.items():
        f.write(f"  {c}: " + ", ".join(f"{i}={v}" for i, v in enumerate(cats)) + "\n")
    f.write("\n=== Tree (text) ===\n" + export_text(tree, feature_names=feat, decimals=2))
    f.write("\n=== Plain-English rules ===\n")
    for _, r in rules.iterrows():
        v = f"{r[val_col]:.1%} YES rate" if IS_CLF else f"average {r[val_col]:,.0f}"
        f.write(f"IF {r['rule (IF ...)']} THEN {v} ({r['training rows']} rows) {r['caution']}\n")
print("Saved: decision_tree_model.joblib and decision_tree_rules.txt")
# Later: bundle = joblib.load("decision_tree_model.joblib"); bundle["model"].predict(new_df[feature_cols])

## Final Checklist

- [ ] TASK, target, and inputs set correctly; IDs and leakage columns excluded
- [ ] Baseline beaten clearly; default (unlimited) tree's overfitting gap seen and fixed
- [ ] Depth and leaf size tuned with cross-validation; shallowest tree within one spread chosen
- [ ] Test set used **once**
- [ ] (Classification) Threshold picked from out-of-fold expected value, not 0.5
- [ ] Rules table reviewed; ⚠ small leaves not used as policy
- [ ] Permutation importance agrees with the story the rules tell
- [ ] Stability check run; first split is stable (or rules presented as hypotheses)
- [ ] Rules shared with a domain expert: do they make sense?
- [ ] Model and rules file saved